In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
#load the data
user_data = pd.read_csv('UserData.csv')
user_historical_view = pd.read_csv('UserHistoricalView.csv')
book_data = pd.read_csv('bookData.csv',encoding='ISO-8859-1')


In [25]:
print("This is UserData")
print(user_data.head())

This is UserData
   userid                        location   age
0   11676                   n/a, n/a, n/a   NaN
1   16795   mechanicsville, maryland, usa  47.0
2   22625         lynbrook, new york, usa  54.0
3   35859          duluth, minnesota, usa   NaN
4   95359  charleston, west virginia, usa  33.0


In [23]:
print("This is UserHistoricalView")
print(user_historical_view.head())

This is UserHistoricalView
   userid       isbn
0   11676   60938455
1   11676  316096199
2   11676  316569321
3   11676  312195516
4   11676  345361792


In [27]:
print("This is bookData")
print(book_data.head())

This is bookData
        isbn              booktitle        bookauthor  yearofpublication  \
0  440234743          The Testament      John Grisham               1999   
1  971880107            Wild Animus      Rich Shapero               2004   
2  345417623               Timeline  MICHAEL CRICHTON               2000   
3  446310786  To Kill a Mockingbird        Harper Lee               1988   
4  671027360      Angels and Demons         Dan Brown               2001   

                    publisher  \
0                        Dell   
1                     Too Far   
2            Ballantine Books   
3  Little Brown &amp; Company   
4                 Pocket Star   

                                            Synopsis  
0  The Testament is a typical John Grisham novel...  
1  Wild Animus tracks the reckless quest of Ranso...  
2  While working at the site of an archaeological...  
3  To Kill a Mockingbird novel by American author...  
4  When Harvard symbologist Robert Langdon discov...

In [29]:
#merge the data
merged_data = pd.merge(user_data, user_historical_view, on='userid')
print("This is merged_data")
print(merged_data.head())



This is merged_data
   userid       location  age       isbn
0   11676  n/a, n/a, n/a  NaN   60938455
1   11676  n/a, n/a, n/a  NaN  316096199
2   11676  n/a, n/a, n/a  NaN  316569321
3   11676  n/a, n/a, n/a  NaN  312195516
4   11676  n/a, n/a, n/a  NaN  345361792


In [30]:
user_book_data = pd.merge(merged_data, book_data[['isbn', 'Synopsis']], on='isbn')
print("This is user_book_data")
print(user_book_data.head())

This is user_book_data
   userid                location  age       isbn  \
0   11676           n/a, n/a, n/a  NaN   60938455   
1   11676           n/a, n/a, n/a  NaN  316096199   
2   11676           n/a, n/a, n/a  NaN  316569321   
3   11676           n/a, n/a, n/a  NaN  312195516   
4   35859  duluth, minnesota, usa  NaN  312195516   

                                            Synopsis  
0  Fast Food Nation: The Dark Side of the All-Ame...  
1  Lucky describes her experience of being raped ...  
2  Powerfully acted adaptation of Janet Fitch's b...  
3  Dinah opens the story by recounting for reader...  
4  Dinah opens the story by recounting for reader...  


In [32]:
null_values = user_book_data.isnull().sum()
print(null_values)

userid       0
location     0
age         32
isbn         0
Synopsis     1
dtype: int64


In [7]:
# Step 3: Handle NaN values in Synopsis
user_book_data['Synopsis'] = user_book_data['Synopsis'].fillna('')


In [60]:
# Step 4: Generate User Profiles by Concatenating Synopses
user_profiles = user_book_data.groupby('userid')['Synopsis'].apply(lambda x: ' '.join(map(str, x))).reset_index()
user_profiles.to_csv('Part1_File1_Profile_Group[group_10].csv', index=False)
print(user_profiles)

   userid                                           Synopsis
0   11676  Fast Food Nation: The Dark Side of the All-Ame...
1   16795  The Secret Life of Bees is a fictional story s...
2   22625  The plot follows Harry's second year at Hogwar...
3   35859  Dinah opens the story by recounting for reader...
4   95359  A Prayer for Owen Meany is a captivating novel...
5  104636  The fourth movie in the Harry Potter franchise...
6  110912  A Prayer for Owen Meany is a captivating novel...
7  204864  Dinah opens the story by recounting for reader...
8  271448  The fourth movie in the Harry Potter franchise...


In [42]:
# Vectorize Text Data
vectorizer = TfidfVectorizer(stop_words='english')
user_tfidf = vectorizer.fit_transform(user_profiles['Synopsis'])
book_tfidf = vectorizer.transform(book_data['Synopsis'].fillna(''))

TfidfVectorizer(stop_words='english')


In [10]:
# Step 5: Calculate Cosine Similarity
cosine_sim_matrix = cosine_similarity(user_tfidf, book_tfidf)

In [61]:
# Convert to DataFrame for easier handling
sim_matrix_df = pd.DataFrame(cosine_sim_matrix, index=user_profiles['userid'], columns=book_data['isbn'])
# Save the similarity matrix to a CSV file
sim_matrix_df.to_csv('Part1_File2_SimMatrix_Group[group_10].csv')
print(sim_matrix_df.head())

isbn    440234743  971880107  345417623  446310786  671027360  60976845  \
userid                                                                    
11676    0.063264   0.039369   0.330709   0.066397   0.070302  0.032523   
16795    0.032116   0.036076   0.000000   0.069560   0.010132  0.044792   
22625    0.007562   0.067184   0.015992   0.031682   0.067983  0.067970   
35859    0.081105   0.030946   0.003110   0.035456   0.000000  0.007795   
95359    0.055223   0.031610   0.029339   0.393171   0.000000  0.034231   

isbn    044021145X  60938455  446672211  142000205  ...  375727345  312924585  \
userid                                              ...                         
11676     0.030166  0.285650   0.281993   0.068107  ...   0.042171   0.077770   
16795     0.028888  0.000000   0.002229   0.057189  ...   0.045627   0.008642   
22625     0.038761  0.011020   0.020323   0.082731  ...   0.120174   0.069419   
35859     0.426774  0.000000   0.017549   0.026038  ...   0.020729   

In [48]:
# Step 6: Find Unread Books for Each User
unread_books = {}
for user in user_data['userid']:
    read_books = user_historical_view[user_historical_view['userid'] == user]['isbn'].tolist()
    unread_books[user] = [isbn for isbn in book_data['isbn'] if isbn not in read_books]


In [13]:
# Step 7: Get Top 5 Recommendations for Each User
recommendations = []
for user in sim_matrix_df.index:
    user_unread_books = unread_books[user]
    user_similarities = sim_matrix_df.loc[user, user_unread_books]
    top_5_books = user_similarities.nlargest(5)
    for isbn, similarity in top_5_books.items():
        book_title = book_data.loc[book_data['isbn'] == isbn, 'booktitle'].values[0]
        recommendations.append({
            'User ID': user,
            'Book’s ISBN': isbn,
            'Book’s Title': book_title,
            'Similarity': round(similarity, 3)
        })


In [56]:
# Convert to DataFrame
recommendations_df = pd.DataFrame(recommendations)
print(recommendations_df.head())

   User ID Book’s ISBN                                       Book’s Title  \
0    11676   590353403     Harry Potter and the Sorcerer s Stone (Book 1)   
1    11676    60987103  Wicked: The Life and Times of the Wicked Witch...   
2    11676   316284955       White Oleander : A Novel (Oprah s Book Club)   
3    11676   316776963                             Me Talk Pretty One Day   
4    11676   439064872   Harry Potter and the Chamber of Secrets (Book 2)   

   Similarity  
0       0.245  
1       0.140  
2       0.129  
3       0.129  
4       0.122  


In [62]:
# Formatting for better CSV output
top_recommendations_df = recommendations_df.sort_values(by=['User ID', 'Similarity'], ascending=[True, False])
print(recommendations_df.head(20))

    User ID Book’s ISBN                                       Book’s Title  \
0     11676   590353403     Harry Potter and the Sorcerer s Stone (Book 1)   
1     11676    60987103  Wicked: The Life and Times of the Wicked Witch...   
2     11676   316284955       White Oleander : A Novel (Oprah s Book Club)   
3     11676   316776963                             Me Talk Pretty One Day   
4     11676   439064872   Harry Potter and the Chamber of Secrets (Book 2)   
5     16795   671003755                                 She s Come Undone    
6     16795    60502258  The Divine Secrets of the YaYa Sisterhood: A N...   
7     16795   345337662                         Interview with the Vampire   
8     16795   375726403                                       Empire Falls   
9     16795   446606812                                Message in a Bottle   
10    22625   590353403     Harry Potter and the Sorcerer s Stone (Book 1)   
11    22625   439136350  Harry Potter and the Prisoner of Azkaba

In [64]:
# Step 8: Store the Result
group_no = 10  # Group number
output_filename = f'Part1_File3_Recommendation_Group[group_no].csv'
recommendations_df.to_csv(output_filename, index=False, float_format='%.3f')

In [17]:
# Display the Recommendations with User Spacing
print("Top 5 Recommendations for Each User:\n")
last_user_id = None
for _, row in recommendations_df.iterrows():
    if last_user_id and row['User ID'] != last_user_id:
        print()  # Add a blank line for separation
    print(f"User ID: {row['User ID']}")
    print(f"  Book’s ISBN: {row['Book’s ISBN']}")
    print(f"  Book’s Title: {row['Book’s Title']}")
    print(f"  Similarity: {row['Similarity']:.3f}")
    last_user_id = row['User ID']

Top 5 Recommendations for Each User:

User ID: 11676
  Book’s ISBN: 590353403
  Book’s Title: Harry Potter and the Sorcerer s Stone (Book 1)
  Similarity: 0.245
User ID: 11676
  Book’s ISBN: 60987103
  Book’s Title: Wicked: The Life and Times of the Wicked Witch of the West
  Similarity: 0.140
User ID: 11676
  Book’s ISBN: 316284955
  Book’s Title: White Oleander : A Novel (Oprah s Book Club)
  Similarity: 0.129
User ID: 11676
  Book’s ISBN: 316776963
  Book’s Title: Me Talk Pretty One Day
  Similarity: 0.129
User ID: 11676
  Book’s ISBN: 439064872
  Book’s Title: Harry Potter and the Chamber of Secrets (Book 2)
  Similarity: 0.122

User ID: 16795
  Book’s ISBN: 671003755
  Book’s Title: She s Come Undone 
  Similarity: 0.148
User ID: 16795
  Book’s ISBN: 60502258
  Book’s Title: The Divine Secrets of the YaYa Sisterhood: A Novel
  Similarity: 0.124
User ID: 16795
  Book’s ISBN: 345337662
  Book’s Title: Interview with the Vampire
  Similarity: 0.100
User ID: 16795
  Book’s ISBN: 37572

FileNotFoundError: [Errno 2] No such file or directory: 'Part2_File3_Recommendation_Group10.csv'